In [ ]:
from openai import OpenAI
client = OpenAI(api_key="""YOUR API KEY""")

position = "<TYPE IN YOUR JOB POSITION>"

run_basic_bool = False  # set to True if you only want to run basic revision (takes priority over comprehensive & standard)
run_comprehensive_bool = False  # set to True if you want to run comprehensive revision
resume_combiner = False  # set to True if you want to run all three revisions and combine into a single resume

userinput_resume = """
  User resume:```
<COPY PASTE YOUR RESUME (IN TEXT FORMAT), LINKEDIN EXPERIENCE, OR ANY OTHER RELEVANT JOB EXPERIENCES TO THE POSITION THAT YOU'RE APPLYING TO>
```
  --------------------------------------------------------------------------------------------------------
  Job posting:'''
<COPY PASTE THE JOB POSTING (IN TEXT FORMAT) HERE>
  '''"""

def run_basic(client, position, userinput):
  system = f"""
  Rewrite my resume for the role of {position}. My resume is delimited by three backticks, and the Job posting is delimited by three single quotation marks.
  Follow the following rules:
  1. DO NOT MAKE UP EXPERIENCE THAT IS NOT INCLUDED IN THE ORIGINAL USER RESUME.
  2. Do not use markdown formatting anywhere.
  3. Each job experience must have exactly two bullet points, each bullet point must be relevant to the job position.
  4. Use confident, achievement-focused language.
  5. Quantify results wherever possible (numbers, %, impact)
  6. Remove weak or generic phrases.
  7. Match the tone of modern tech/startup resumes.
  """

  response = client.responses.create(
      model="gpt-5",
      reasoning={"effort": "low"},
      instructions=system,
      input=userinput
  )

  print(response.output_text + "\n\n\n" + "*" * 80)
  return response.output_text


def run_standard(client, position, userinput):
  system = f"""
You are an expert recruiter and resume writer. You are hiring candidates for the role of {position}, and today you have been approached by an applicant
who is asking for your expertise in resumes to match what you actually scan for in 10 seconds.

My resume is delimited by three backticks, and the Job posting is delimited by three single quotation marks.
Follow the following rules:

1. DO NOT MAKE UP EXPERIENCE THAT IS NOT INCLUDED IN THE ORIGINAL USER RESUME.
2. Rewrite my experience using numbers, impact, and outcomes. Remove responsibilities. Keep only results.
3. Each job experience must have exactly two bullet points, each bullet point must be relevant to the job position.
4. Use confident, achievement-focused language.
5. Quantify results wherever possible (numbers, %, impact)
6. Remove weak or generic phrases.
7. Match the tone of modern tech/startup resumes.
8. Optimize my resume for ATS keywords for this job description without sounding robotic.”
9. Do not use markdown formatting anywhere.
10. Cut my resume down to one page while increasing clarity and relevance."""

  response = client.responses.create(
      model="gpt-5",
      reasoning={"effort": "low"},
      instructions=system,
      input=userinput
  )

  print(response.output_text + "\n\n\n" + "*" * 80)
  return response.output_text


def run_comprehensive():
  system = f"""
You are an expert recruiter and resume writer. Help me improve my resume for {position} so it clearly reflects my real experience, sounds like me, and positions me strongly for the roles I’m targeting, increasing my chances of landing interviews.

NON-NEGOTIABLE RULES:
1. Do not invent, exaggerate, or assume anything (experience, metrics, employers, titles, tools, certifications, dates).
2. If information is missing or unclear, ask me before writing.
3. Keep my voice human and confident. No corporate fluff.
4. Do not copy wording from job postings. Translate my real experience into relevant language.
5. Optimize for humans and ATS: clear structure, clean formatting, keyword alignment without stuffing.

{userinput}

STEP 1: After reviewing everything, summarize:
* The top 8–12 skills/keywords across the postings
* The 3–5 outcomes the hiring manager likely cares most about
* The biggest gaps or weaknesses in my resume for these roles

STEP 2: Deliver the following:
A) A tailored, ATS-friendly resume (no tables, no columns) including:
  - A strong headline and 2-line summary
  - Core Skills under each relevant job (10–14 max)
  - Experience bullets written as: Action + Scope + Outcome
  - Tight bullets (1–2 lines), most relevant first
B) A change log explaining what you changed and why
C) A truth check list of anything that still needs my confirmation

FORMATTING
* Use standard headings only: Summary, Core Skills, Experience, Education, Certifications
* Present tense for current roles; past tense for previous roles
* Use strong verbs and specific outcomes
* No buzzwords, clichés, graphics, icons, or heavy design
"""
  response = client.responses.create(
      model="gpt-5",
      reasoning={"effort": "low"},
      instructions=system,
      input=userinput
  )

  print(response.output_text + "\n\n\n" + "*" * 80)
  return response.output_text


def combine_resumes(client, position, resumes):
  system = f"""
You are an expert resume writer. Today your job is to look over three resumes written for the same position of {position} and combine them together to a single resume for the position.
Follow the following rules:

1. DO NOT MAKE UP EXPERIENCE THAT IS NOT INCLUDED IN THE ORIGINAL USER RESUME.
2. Each job experience must have exactly two bullet points, each bullet point must be relevant to the job position.
3. Use confident, achievement-focused language and remove weak or generic phrases. Match the tone of modern tech/startup resumes.
4. Optimize the resume for ATS keywords for this job description without sounding robotic.
5. Do not use markdown formatting anywhere.
6. Cut the resume down to one page while increasing clarity and relevance."""

  input_array = [
          {
              "role": "developer",
              "content": system
          }
  ]
  for resume in resumes:
    input_array.append({
      "role": "user",
      "content": resume
    })


  response = client.responses.create(
    model="gpt-5",
    reasoning={"effort": "low"},
    input=input_array
  )

  print(response.output_text)

YOUR RESUME TEXT GOES HERE

In [ ]:
if resume_combiner:
  combine_resumes(client, position, [run_old(client, position, userinput_resume), run_new_alternate(client, position, userinput_resume), run_new(client, position, userinput_resume)])
elif run_basic_bool:
  run_basic(client, position, userinput_resume)
elif run_comprehensive_bool:
  run_comprehensive(client, position, userinput_resume)
else:
  run_standard(client, position, userinput_resume)